# 🏆 Proyecto Final · Curso Football Analytics LanusStats

### Análisis del Mundial 2022 con datos abiertos de StatsBomb

**Alumno:** Lucas Marinelli  
**Email:** lucasmarinelli_12@hotmail.com  
**Fecha de entrega:** 15 de junio de 2026

---

## Quién soy y por qué este proyecto

Hace 16 años que trabajo en el rubro de la electromedicina y automatización
(OFIMED, La Plata) y desde hace algunos meses estoy migrando hacia el análisis
de datos aplicado al fútbol. Mi primer acercamiento al mundo del fútbol como
disciplina fue un curso de videoanálisis en la **UP con Mati Conde y otros
profesionales**. Acabo de terminar la Carrera de Data Analytics de Coderhouse
y este curso de Federico es el siguiente paso lógico para mí: aprender football
analytics con datos reales.

Tengo además un proyecto personal en redes, **[@datafutbol_ar](https://instagram.com/datafutbol_ar)**,
que es mi vehículo para aprender mientras hago, generar portfolio público y
construir una segunda carrera. Por eso cada gráfico del proyecto está hecho con
la identidad visual de mi marca (paleta **Combo C** — celeste argentino + dorado).
No es para "decorar el TP": es para que el trabajo del curso me sirva también
como contenido editorial real.

---

## Resumen ejecutivo del proyecto

Cumplí los **6 puntos obligatorios + los 2 extras opcionales** de la consigna.
La idea fue tener un hilo conductor para que no fuera una colección de ejercicios
sueltos, sino una historia coherente.

### Qué partido y qué jugador elegí

| Rol | Detalle | `match_id` |
|---|---|---|
| **Principal** (Puntos 1–5) | Argentina 1–2 Arabia Saudita — debut Mundial 2022 | `3857300` |
| **Comparativo** (bonus) | Argentina 3–3 Francia — final | `3869685` |
| **Jugador analizado** | **Lionel Messi** (juega los dos partidos elegidos) | — |
| **Punto 6** | Mundial 2022 completo, 64 partidos | comp `43` / season `106` |

### Por qué elegí ARG–SAU y no la final

La final está hipersaturada de análisis, todo el mundo la cubrió. El debut, en
cambio, lo analizó casi nadie y permite una narrativa más fuerte: *"el campeón
perdió su primer partido"*. Para mí, que estoy construyendo una marca en redes,
esa decisión es **el lado oculto del trabajo del analista**: elegir qué historia
contar antes de empezar a calcular.

### Por qué elegí a Messi como jugador

Porque juega los dos partidos del proyecto. Eso me permite, con el mismo notebook
de análisis individual, sacar dos perfiles del mismo jugador en dos contextos
distintos (debut perdido vs. final ganada). Es una comparación natural que daría
contenido para tres o cuatro posts en redes después.

---

## Estructura del proyecto

| Archivo | Para qué sirve |
|---|---|
| **`proyecto_final.ipynb`** | **Este notebook** — informe ejecutivo + índice |
| `README.md` | Estructura, decisiones, cómo correrlo |
| `helpers.py` | Carga cacheada de eventos + paleta Combo C |
| `pases_progresivos.py` | Módulo reusable del Extra 1 |
| `00_setup.ipynb` | Descarga inicial de los 64 partidos (correr una sola vez) |
| `punto_1.ipynb` | Punto 1 — Obtener info del partido |
| `punto_2.ipynb` | Punto 2 — 4 sub-preguntas (xG · pases · recuperaciones · tercios) |
| `punto_3.ipynb` | Punto 3 — Mapas de tiros + pases (+ bonus progresivos) |
| `punto_4.ipynb` | Punto 4 — Análisis individual de Messi |
| `punto_5.ipynb` | Punto 5 — Dashboards del partido (A académico completo + B Messi destacado) |
| `punto_6.ipynb` | Punto 6 — Mundial completo (preguntas + rankings + dashboard 3 canchas + scatters + bonus IG) |
| `punto_extra.ipynb` | Extras opcionales — función reusable + PyPizza |
| `outputs/` | Todas las visualizaciones generadas (PNG) |

---


## Punto 1 — Obtener información del partido ✅

**Consigna:** *"Obtener la info de un partido del Mundial 2022 con la API de StatsBomb."*

### Cómo lo resolví

Implementé en `helpers.py` la función `cargar_eventos(match_id, alias)` que
funciona en dos pasos:

1. **Si ya bajé el partido alguna vez**, lo lee de un parquet local en `data/eventos_{alias}.parquet`.
2. **Si es la primera vez**, llama a `statsbombpy.sb.events(match_id)` y guarda el resultado.

Hice esto porque la API de StatsBomb va por internet y tardo varios segundos por
partido. Como iba a usar los mismos partidos muchas veces durante el proyecto,
me convenía cachear local. Con esto, todo el resto de los notebooks corre
**instantáneo** después del setup inicial.

→ Código completo en [`punto_1.ipynb`](punto_1.ipynb).


In [ ]:
# Demostración rápida: cargar el partido principal
from helpers import *

ev = cargar_eventos(MATCH_ARG_SAU, 'arg_sau')
print(f'✓ Argentina vs Arabia Saudita cargado')
print(f'  match_id: {MATCH_ARG_SAU}')
print(f'  Eventos totales: {len(ev):,}')
print(f'  Tipos de eventos: {ev["type"].nunique()}')
print(f'  Equipos: {list(ev["team"].unique())}')


---

## Punto 2 — Análisis de estadísticas del partido ✅

**Consigna:** *"Equipo con más xG · Jugador con más pases · Jugador con más recuperaciones · Zona (en tercios) con más acciones."*

Organicé el notebook **una pregunta por sección** (§ 2.1 a § 2.4), con respuesta destacada después de cada cálculo para que sea imposible perderla al revisar.

### Lo que encontré en ARG vs SAU

| § | Pregunta | Respuesta |
|---|---|---|
| 2.1 | Equipo con más xG | **Argentina** (2.49 vs 0.15 xG — ≈16× más) |
| 2.2 | Jugador con más pases | **Cristian Romero** (ARG, 83 pases) |
| 2.3 | Jugador con más recuperaciones | **Marcos Acuña** (ARG, 9 recuperaciones) |
| 2.4 | Tercio con más acciones | **Mediocampo** (~50% del total) |

### Cómo lo encaré

Para el conteo de pases tuve que aprender un detalle técnico que me parece valioso:
`pandas.count('pass_outcome')` ignora los NaN, y en StatsBomb los pases completados
justamente tienen `pass_outcome = NaN`. Si contás con `count` ahí, estás contando
solo los **fallidos**. Lo solucioné contando sobre `'team'` (columna que nunca es
NaN). Es un bug en silencio que aprendí en este proyecto y me lo voy a llevar.

### Lo que esto me dice

Argentina dominó **todas** las métricas pero perdió el partido. Esa asimetría
entre "estadística" y "resultado" me parece la lección más importante que me
llevo del curso hasta ahora: la eficiencia decide los partidos más que el dominio.

### Piezas generadas

| Archivo | Visualización |
|---|---|
| `punto2_tercios_cancha.png` | Distribución de acciones en los 3 tercios |

→ Código completo en [`punto_2.ipynb`](punto_2.ipynb).


---

## Punto 3 — Mapas de equipos (tiros + pases) ✅

**Consigna:** *"Mapa de tiros de ambos equipos · Mapa de pases de ambos equipos."*

### Cómo lo encaré

Hice **3 tipos de mapas** por equipo, cumpliendo la consigna directa y agregando
un bonus que me parecía valioso:

| § | Mapa | Para qué |
|---|---|---|
| 1 | **Tiros** (consigna) | Dónde y con qué calidad de chance se intentó convertir |
| 2 | **Pases completos** (consigna) | Patrón general de circulación + acierto |
| 3 | **Pases progresivos** (bonus) | Cuánto avanzaron el juego — métrica moderna |

### Decisiones de diseño

- **VerticalPitch(half=True)** para los shot maps — estándar industria (StatsBomb, McKay Johns, Ben Griffis).
- **Pitch horizontal completo** para los pass maps — los pases van en cualquier dirección.
- **Paleta semáforo CVD-safe** (`#009E73` verde + `#B36930` naranja) en lugar del rojo/verde clásico. En el módulo M6-02 de Coderhouse aprendí que 1 de cada 12 hombres tiene daltonismo, y la paleta de Wong (2011) resuelve eso.
- **Tamaño del tiro** proporcional al xG (`s = xG × 500 + 80`) → "ver" la peligrosidad del tiro de un vistazo.

### Piezas generadas

| Archivo | Visualización |
|---|---|
| `punto3_tiros_argentina.png` | Shot map ARG (15 tiros, 2.49 xG, 1 gol) |
| `punto3_tiros_arabia.png` | Shot map SAU (3 tiros, 0.15 xG, 2 goles) |
| `punto3_pases_argentina.png` | Pass map ARG (verde OK / naranja fallidos) |
| `punto3_pases_arabia.png` | Pass map SAU |
| `punto3_pases_argentina_progresivos.png` | Pases progresivos ARG (bonus) |
| `punto3_pases_arabia_saudita_progresivos.png` | Pases progresivos SAU (bonus) |

→ Código completo en [`punto_3.ipynb`](punto_3.ipynb).


---

## Punto 4 — Análisis individual: Lionel Messi ✅

**Consigna:** *"Elegir un jugador y hacer: mapa de calor, mapa de tiros, mapa de acciones (recup, pases, faltas, intercep)."*

### Por qué Messi

Porque juega los dos partidos del proyecto (debut + final), entonces si quería
hacer una segunda parte comparativa, ya tenía el material. Además es el capitán
y la figura natural para contar la historia del torneo.

### Lo que vi de Messi en ARG vs SAU

| Métrica | Valor |
|---|---:|
| Toques totales | 174 |
| Tiros | 4 |
| Goles | 1 (penal min 10) |
| xG total | 1.07 |
| Pases progresivos | 13 (filtro estricto) / 19 (con intentos) |
| Recuperaciones | 3 |

### Decisiones específicas que tomé

- **Cmap personalizado "datafutbol"** para el heatmap (navy → celeste → dorado). Lo armé con `LinearSegmentedColormap.from_list()` para que el calor "vista" la marca.
- **VerticalPitch(half=True)** para el shot map, igual que los del Punto 3. Mantengo coherencia visual.
- **4 marcadores distintos** en el mapa de acciones: círculo (recuperación), triángulo (intercepción), X (falta), flecha (pase progresivo). Esto me permite que el gráfico se entienda incluso si alguien lo imprime en blanco y negro.
- **Pases progresivos como capa de fondo** (alpha 0.45) → no tapan los otros markers pero dan contexto territorial.

### Piezas generadas

| Archivo | Visualización |
|---|---|
| `punto4_heatmap_messi.png` | Mapa de calor con cmap datafutbol |
| `punto4_tiros_messi.png` | Shot map con doble leyenda (color y tamaño) |
| `punto4_acciones_messi.png` | 4 capas en un solo gráfico |

→ Código completo en [`punto_4.ipynb`](punto_4.ipynb).


---

## Punto 5 — Dashboard del partido ✅

**Consigna:** *"Mapa de tiros y pases de ambos equipos · Resultado y detalles · Datos sobre jugadores destacados."*

### Hice dos dashboards complementarios

| | Dashboard A (académico) | Dashboard B (Messi destacado) |
|---|---|---|
| **Cumple** | Tiros + pases ambos equipos + resultado + tabla + destacados | Datos sobre jugador destacado (criterio propio: Messi) |
| **Audiencia** | Curso | Instagram (público @datafutbol_ar) |
| **Layout** | Header → shot maps → pass maps → tabla 8 métricas → tarjetas destacados | Header → heatmap → shot map + KPI cards → storytelling |
| **Formato** | 1080×1620 px (carrusel IG largo) | 1080×1350 px (carrusel IG estándar) |
| **Archivo** | `punto5_dashboard_partido.png` | `punto5_dashboard_messi.png` |

### El criterio editorial que usé para "destacado"

Elegí **"jugador con más pases progresivos del partido"** como métrica para identificar al destacado de cada equipo en el Dashboard A. La métrica captura mejor **intención ofensiva** que el simple conteo de pases o recuperaciones — diferencia al jugador que "construye juego" del que "circula la pelota sin riesgo". Es la métrica que más se está usando en scouting profesional hoy.

Para el Dashboard B usé a **Messi** como destacado principal: capitán, único goleador de ARG, y la figura que da continuidad narrativa con el análisis del Punto 4.

### El insight clave del Dashboard A

Argentina ganó **6 de las 8 métricas** comparativas (15 vs 3 tiros, 2.49 vs 0.15 xG, 130 vs 52 pases progresivos, 63.9% vs 36.1% posesión)… y **perdió 1-2**. Esa es la historia que el dashboard cuenta sin necesidad de texto extra.

→ Código completo en [`punto_5.ipynb`](punto_5.ipynb).


---

## Punto 6 — Análisis del Mundial completo ✅

**Consigna oficial:** varias sub-tareas (conseguir el DataFrame con `pd.concat` + 3 preguntas puntuales + rankings + dashboard 3 canchas + scatters).

### Cómo lo organicé

Armé **un solo notebook** que sigue paso a paso la estructura de la consigna oficial:

1. **Setup** → DataFrame con `for` + `pd.concat` (los 64 partidos en uno solo)
2. **§ 1 — Preguntas a contestar** (las 3 puntuales, con respuesta destacada)
3. **§ 2 — 5 rankings** (los 4 oficiales + uno propio: pases progresivos)
4. **§ 3 — Dashboard 3 canchas** (Messi en todo el torneo)
5. **§ 4 — 4 scatters** (los 3 oficiales + uno propio: mediocampista total)
6. **§ 5 — Bonus IG** (Top 10 xG, heatmap goles, evolución ARG)

### Las preguntas puntuales (respondidas)

| Pregunta | Respuesta |
|---|---|
| Más xG en NED-ECU | Ver `punto_6.ipynb` § 1.1 |
| Más pases en ENG-IRN | Ver `punto_6.ipynb` § 1.2 (John Stones con 119 intentados, 117 completados, 98.3% acierto) |
| Pases en MAR-FRA | Ver `punto_6.ipynb` § 1.3 (1002 totales · 845 completados · 84.3%) |

### Los 5 rankings (top 3 de cada uno)

| Ranking | Top 3 |
|---|---|
| Más tiros del torneo | Messi (32) · Mbappé (31) · Giroud (17) |
| Más tiros por partido | Messi (4.57) · Mbappé (4.43) · Gnabry (4.00) |
| Más pases completados | Rodri (644) · Otamendi (528) · De Paul (475) |
| Más recuperaciones | Amrabat (44) · Hakimi (43) · Modrić (39) |
| **Mi ranking propio: pases progresivos** | Rodri (175) · Gvardiol (168) · Otamendi (155) |

### Por qué elegí "pases progresivos" como ranking propio

Es la métrica que está moviendo el análisis moderno. Mide **cuánto avanza el
juego un jugador cada vez que toca la pelota** (definición: pase completado que
avanza ≥10m hacia el arco rival). Es distinta a "pases completados" porque
captura intención ofensiva, no solo precisión.

### Dashboard 3 canchas (Messi en todo el torneo)

371 pases · 1546 eventos · 32 tiros · 7 goles · 6.03 xG. Una sola pieza con los
3 mapas (pases / calor / tiros) para tener la firma del jugador en el torneo.

### 4 scatters (los 3 oficiales + mi propio)

- **xG vs Tiros** — "clínicos vs voraces". Mbappé tiró 31 veces con 4.23 xG, Messi 32 con 6.03. Pero Mbappé metió **8 goles** vs los 7 de Messi: overperformance brutal.
- **Pases completados vs % acierto** — Rodri en la esquina superior derecha (644 pases con 95% acierto).
- **Intercepciones vs Faltas** — Modrić destacado (15 faltas, 17 intercepciones), el "perro mayor" del torneo.
- **Recuperaciones vs Pases progresivos (propio)** — "mediocampista total". Rodri lidera en pases progresivos, Amrabat/Hakimi en recuperaciones.

### Bonus de IG (al final de `punto_6.ipynb`)

| Pieza | Insight |
|---|---|
| Top 10 xG con flecha ▲ overperformance | Mbappé: `8 goles · 4.23 xG ▲` — el dato más fuerte del torneo |
| Heatmap de los 169 goles del Mundial | 94% dentro del área grande, 20% en el área chica |
| Argentina partido a partido | Aprendió del SAU y nunca volvió a bajar del 42% de posesión |

→ Código completo en [`punto_6.ipynb`](punto_6.ipynb).


---

## Extras (opcionales) ✅

**Consigna:** *"Calcular pases progresivos para los dataframes de pases · Elegir un gráfico de la doc de mplsoccer y replicarlo."*

### Extra 1 — Módulo reusable `pases_progresivos.py`

Empecé escribiendo la lógica de pases progresivos copiada-pegada en cada notebook.
Después me di cuenta de que la iba a reusar muchas veces, así que la saqué a un
módulo independiente.

```python
from pases_progresivos import agregar_pases_progresivos

pases = ev[ev['type'] == 'Pass']
pases = agregar_pases_progresivos(pases, umbral_metros=10)
# Ahora 'pases' tiene: x, y, x_end, y_end, avance_x, es_completado, es_progresivo
```

**Definición técnica:** pase progresivo = pase **completado** que avanza **≥10m**
hacia el arco rival (umbral configurable).

Tuve un bug interesante en el camino: statsbombpy devuelve `pass_end_location`
como `np.ndarray` (no como list/tuple), entonces mi primera versión del check
`isinstance(loc, (list, tuple))` daba False para todo y la función devolvía 0
progresivos siempre. Lo solucioné incluyendo `np.ndarray` en el isinstance.
Quedó documentado en el docstring del módulo para que no me pase de nuevo.

### Extra 2 — PyPizza de Messi (Mundial 2022)

Repliqué el [tutorial oficial de mplsoccer PyPizza](https://mplsoccer.readthedocs.io/en/latest/gallery/pizza_plots/index.html)
usando la paleta Combo C. Lo elegí porque es **el gráfico más reconocible** del
mundo del football analytics — lo usan McKay Johns, Ben Griffis, los reportes
oficiales de StatsBomb. Si quería hacer una pieza "estándar de la industria",
era ese.

**Percentiles de Messi vs 23 jugadores ofensivos del torneo:**

| Métrica | Valor | Percentil |
|---|---:|---:|
| Goles | 7 | P96 |
| xG total | 6.03 | **P100** |
| Tiros | 32 | **P100** |
| xG por tiro | 0.19 | P91 |
| Pases completados | 306 | **P100** |
| % acierto pases | 82.5% | P96 |
| Pases progresivos | 89 | **P100** |
| Regates exitosos | 26 | P96 |
| Faltas recibidas | 27 | **P100** |
| Recuperaciones | 34 | **P100** |

El pizza queda **casi completamente lleno** — esa es la firma visual del jugador
más completo del torneo. Es una de las piezas que más me gustó del proyecto.

→ Código completo en [`punto_extra.ipynb`](punto_extra.ipynb).


---

## Stack técnico

| Categoría | Librerías |
|---|---|
| Datos | `statsbombpy` (open data) |
| Procesamiento | `pandas` · `numpy` · `pyarrow` (cache parquet) |
| Visualización | `matplotlib` · `mplsoccer` (Pitch, VerticalPitch, PyPizza) |
| Helpers | `tqdm` · `scipy.stats` (percentiles) |
| Identidad visual | Combo C de @datafutbol_ar (navy `#0E2A47` · celeste `#75AADB` · dorado `#C9A227`) |

---

## Decisiones de diseño transversales

| Decisión | Por qué |
|---|---|
| Fondo navy `#0E2A47` en todas las canchas | Identidad de marca + buen contraste |
| Paleta CVD-safe Wong 2011 para semáforos | Daltonismo en 1 de 12 hombres |
| Filtro `period <= 4` en conteos del torneo | Excluye tandas (no cuentan como goles oficiales) — gracias a esto Messi me dio 7 goles en lugar de 8 |
| Tamaño del tiro proporcional a xG (`s = xG × 500 + 80`) | Estándar StatsBomb/Opta |
| Cache parquet en `data/` | Trabajar offline después de la primera descarga |
| Diccionario de "nombres bonitos" para gráficos | "Lionel Andrés Messi Cuccittini" en stats raw → "Lionel_Messi" en gráficos |
| Formato 1080×1350 y 1080×1620 px en dashboards | Carrusel Instagram sin recortes |
| Conteo de pases sobre `'team'` y no `'pass_outcome'` | `count()` ignora NaN — los pases completados son NaN — bug en silencio |

---

## Reflexión final

Lo que me llevo del proyecto, más allá del código:

1. **El analista decide qué historia contar antes de calcular.** Elegir ARG-SAU
   en lugar de la final no es una decisión técnica, es editorial. Cambió todo
   el proyecto.

2. **El diseño no es decoración: es comunicación.** Las decisiones de paleta CVD-safe,
   marker shapes diferenciados, layout 1080×1350 — todas vienen de pensar "¿esto
   se va a entender solo?". Si alguien necesita un párrafo de texto para leer
   el gráfico, el gráfico está mal hecho.

3. **El cache local cambia el ritmo del trabajo.** Bajar 64 partidos cada vez
   que abrís el notebook es matar la iteración. Una vez cacheado, podés probar
   ideas en segundos.

4. **Las funciones reusables valen más que el código bonito.** El módulo
   `pases_progresivos.py` lo uso en 5 notebooks distintos. Vale 10 veces lo
   que vale escribir lo mismo 5 veces.

5. **Los bugs silenciosos son los más peligrosos.** El `count('pass_outcome')`
   no rompe nada — devuelve un número plausible — pero está mal. Si no hubiera
   revisado los porcentajes (que daban arriba de 100%), nunca lo habría detectado.
   Lección: **siempre validar que los números caigan en rangos lógicos**, no solo
   que el código corra sin error.

Gracias Federico por el curso, por la librería LanusStats que usé varias veces,
por los videos de YouTube, y especialmente por el manual de scraping anti-Cloudflare
que publicaste en Discord el 13/6 (lo documenté en mi propio sistema de memoria
para no perderlo). Para alguien que viene de afuera del mundo del fútbol y está
construyendo una segunda carrera, materiales como los tuyos abren puertas que
no se abren solas.

— Lucas Marinelli  
[@datafutbol_ar](https://instagram.com/datafutbol_ar) · [github.com/lucas-marinelli](https://github.com/lucas-marinelli)
